# Pairwise Sequence Alignment — Guided Exercises

**AMLG | Global and local sequence alignment, read mapping, and introductory variant calling**

This notebook is designed for a **live teaching session**. We first learn why alignment is useful and how it works, then apply the same ideas in a small simulated sequencing workflow. Work through the short explanations and examples before completing each ❓ exercise.

### How to use this notebook

1. Run cells **from top to bottom** with **Shift + Enter**.
2. Before each exercise, read **Task**, **Hints**, and **What this teaches**.
3. Follow the `# TODO` comments in the code cell. Starter code stays incomplete on purpose; commented answers appear below `# --- Solution ---`.
4. For the classroom demo, uncomment the relevant solution lines and remove any placeholder `pass` before running the exercise.
5. Compare the output with **Expected outcome**, then continue to the next section.

**Learning route:** Needleman–Wunsch (global) → Smith–Waterman (local) → sample reads → add noise → map reads → build pileups → majority-vote SNP calling.

> **Teaching note:** The original theory, formula, examples, and exercises are retained. Additional explanations and commented solutions are there to support instruction, not to replace the original material.


### Lesson objectives

By the end, you should be able to explain **why** we align sequences, distinguish **global** from **local** alignment, read a dynamic-programming scoring matrix and a traceback path, and describe how mapped reads lead to a simple SNP candidate call.

**For the instructor:** Show the `AG` vs `A` matrix example before introducing the full Needleman–Wunsch formula. For coding exercises, explain the task first, let students try the TODO, and only then reveal the commented solution.


# The Needleman-Wunsch Algorithm

### Before the formula: what are we computing?

Suppose `x = "AG"` and `y = "A"`. A possible end-to-end alignment is:

```text
X: A G
Y: A -
   |  
```

With **match = +1**, **mismatch = −1**, **gap = −1**, its score is `+1 − 1 = 0`.

A **dynamic programming (DP) matrix** finds the highest-scoring alignment without enumerating every complete alignment. A cell `F(i,j)` is the **best score for aligning the first `i` characters of X with the first `j` characters of Y**. In particular, `(0,0)` means two empty prefixes.

| Symbol | Meaning | Example |
|---|---|---|
| `i` | Number of X characters already considered (matrix row) | `i=2` means `AG` |
| `j` | Number of Y characters already considered (matrix column) | `j=1` means `A` |
| `xᵢ`, `yⱼ` | Current characters at those prefix ends | `x₂=G`, `y₁=A` (mathematical 1-based subscripts) |
| `s(xᵢ,yⱼ)` | Substitution score: +1 if equal, −1 otherwise | `s(A,A)=+1` |
| `d` | Penalty for introducing one gap | `d=−1` |
| `F(i,j)` | Best score for these two prefixes | `F(1,1)=1` |

**Read the three terms as three choices for the *last alignment column*:**

| Candidate | Where the score comes from | Alignment column added |
|---|---|---|
| `F(i−1,j−1) + s(xᵢ,yⱼ)` | diagonal ↖ | `xᵢ` aligned with `yⱼ` (match/mismatch) |
| `F(i−1,j) + d` | above ↑ | `xᵢ` aligned with `-` (gap in Y) |
| `F(i,j−1) + d` | left ← | `-` aligned with `yⱼ` (gap in X) |

`max` simply chooses whichever of these three possibilities produces the greatest score. **The original formula is shown next, unchanged.**

The Needleman-Wunsch algorithm finds the optimal *global* alignment (end-to-end) between two sequences.
It consists of the following steps:

1. Construct the grid
2. Choose a scoring system (here we use the original scoring system assigning a score $s$ of 1 to a match, a score $s$ of -1 to a mismatch, and a score $d$ of -1 to a gap/indel)
3. Fill in the scoring matrix $F$ (and construct the pointer matrix on the way)
4. Use the pointer matrix to perform a traceback to recover the optimal warping path

The scores in the scoring matrix $F$ are computed using the already existing scores in the left, top-left, and top cells:

$$
F(i,j)=\max
\begin{cases}
    F(i-1,j-1)+s(x_i,y_i),\\
    F(i-1,j)+d,\\
    F(i,j-1)+d.
\end{cases}
$$

The following image illustrates the scoring and pointer matrices.

<img src="images/nw_matrix.png" width="500" />

### Worked example: fill the entire matrix by hand

For `X = AG`, `Y = A`, the matrix has **3 rows** (`empty`, `A`, `AG`) and **2 columns** (`empty`, `A`).

**Step 1 — initialize empty-prefix alignments.** Each leading gap costs −1:

| X prefix \ Y prefix | empty (j=0) | A (j=1) |
|---|---:|---:|
| empty (i=0) | 0 | −1 |
| A (i=1) | −1 | ? |
| AG (i=2) | −2 | ? |

**Step 2 — compute `F(1,1)` (align `A` with `A`).**

- Diagonal: `F(0,0) + s(A,A) = 0 + 1 = 1` **← largest**
- Up: `F(0,1) + d = −1 − 1 = −2`
- Left: `F(1,0) + d = −1 − 1 = −2`

So **`F(1,1) = 1`**, and we record a **↖ pointer**.

**Step 3 — compute `F(2,1)` (align prefixes `AG` and `A`).**

- Diagonal: `F(1,0) + s(G,A) = −1 − 1 = −2`
- Up: `F(1,1) + d = 1 − 1 = 0` **← largest**
- Left: `F(2,0) + d = −2 − 1 = −3`

So **`F(2,1) = 0`**, and we record an **↑ pointer**.

| X prefix \ Y prefix | empty | A |
|---|---:|---:|
| empty | 0 | −1 |
| A | −1 | **1 ↖** |
| AG | −2 | **0 ↑** |

**Final global-alignment score = bottom-right value = `0`.**

### Traceback: turn the scores into actual letters

The scores say *how good* each partial alignment is. The **pointers** say *which previous cell* supplied the winning score. We start at the **bottom-right** and follow pointers **backward**:

1. At `F(2,1)`, pointer **↑** means consume X's `G`, insert a gap in Y: write `G / -`. Move to `F(1,1)`.
2. At `F(1,1)`, pointer **↖** means align X's `A` with Y's `A`: write `A / A`. Move to `F(0,0)`.
3. Stop at `(0,0)`. We reconstructed columns **from last to first**, so reverse their order.

```text
Traceback order:     (G,-) ← (A,A)
Final alignment X:    A G
Final alignment Y:    A -
Total score:          +1 -1 = 0
```

**What if two candidates tie?** Multiple optimal alignments can exist; an implementation uses a tie-breaking rule (or reports multiple paths).

We can use the `make_align()` function from our `alignment` module to generate an `align()` function that performs the Needleman-Wunsch algorithm.

In [ ]:
import alignment as aln

# Two DNA sequences to compare from beginning to end.
x = "GATTACA"
y = "CATACCCA"

# Create a GLOBAL aligner (Needleman-Wunsch).
align = aln.make_align(mode="global")
# Run alignment and unpack: aligned letters, path coordinates, and DP scores.
x_aln, y_aln, warp_path_x, warp_path_y, scoring_mat = align(seq_a=x, seq_b=y)

### Reading the five outputs: letters versus positions

```python
x_aln, y_aln, warp_path_x, warp_path_y, scoring_mat = align(seq_a=x, seq_b=y)
```

- **`x_aln` and `y_aln`**: the **aligned characters**, including `-` if the algorithm inserts gaps. These are the sequences we want to display.
- **`warp_path_x` and `warp_path_y`**: numerical coordinates describing **where the selected alignment travels in the algorithm's scoring grid**. They are used for plotting or recovering mapping locations.
- **`scoring_mat`**: the DP scores, not the alignment itself.

For instance, `X = ATG` versus `Y = AG` can align as:

```text
Alignment column:     1 2 3
X (aligned letters):   A T G
Y (aligned letters):   A - G
X base index:          0 1 2
Y base index:          0 - 1
```

A **conceptual full DP-grid path** in `(row, column)` coordinates is `(0,0) → (1,1) → (2,1) → (3,2)`; the second step advances X but not Y, which means **a gap in Y**.

**Important distinction:** this *full grid path* is not necessarily the exact array returned by the course's `warp_path_x` / `warp_path_y`. The original notebook does not include `alignment.py`, so we should **inspect the actual returned arrays** instead of assuming an undocumented indexing and endpoint convention. The next diagnostic cell does exactly that, if the course module is available.

In [ ]:
# Optional diagnostic: verify how your course alignment module represents the path.
print("Aligned X:", x_aln)
print("Aligned Y:", y_aln)
print("Path X:", warp_path_x)
print("Path Y:", warp_path_y)
print("Matrix shape:", scoring_mat.shape)

We can also plot the alignment together with the scoring matrix:

In [ ]:
import matplotlib.pyplot as plt  # type: ignore

plt.imshow(X=scoring_mat)
plt.plot(warp_path_y, warp_path_x, "w")
plt.xticks(ticks=range(len(y)), labels=y)
plt.yticks(ticks=range(len(x)), labels=x)
plt.show()

# The Smith-Waterman Algorithm

The Smith-Waterman algorithm finds the optimal *local* alignment between two sequences.
It is a variation of the Needleman-Wunsch algorithm.

The three main differences are:

|                | Smith-Waterman algorithm                                | Needleman-Wunsch algorithm                                                 |
|----------------|---------------------------------------------------------|----------------------------------------------------------------------------|
| Initialization | First row/column = 0                                    | First row/column subject to gap penalty                                    |
| Scoring        | Negative scores are set to 0                            | Scores can be negative                                                     |
| Traceback      | Begin with the highest score, end when 0 is encountered | Begin with the cell at the lower right of the matrix, end at top left cell |

### Why does local alignment behave differently?

Compare `X = GATTACA` with `Y = NNNNGATTACANNNN`. If we want the **best matching region** rather than forcing the `N` characters to align, we use **Smith–Waterman**.

Smith–Waterman applies the same three candidates but also allows **0**:

$$F(i,j)=\max\big(0,\ F(i-1,j-1)+s(x_i,y_j),\ F(i-1,j)+d,\ F(i,j-1)+d\big).$$

A negative score is replaced by **0** so a poor earlier region does not drag down a promising new local match. Therefore traceback starts at the **highest-scoring cell anywhere** and stops once it reaches **0**. Unlike global alignment, local alignment does **not** have to include both sequences' ends.

**Teaching check:** When mapping a short read to a long reference, do we usually want full-length global alignment between both entire strings? **No** — the read is expected to match just part of the reference.

Again, with our `alignment` module, we can perform the local alignment between the sequences `GATACCA` and `NNNNNNNNNNGATTACANNNNNNNNGATNNNNNNNNN`.

In [ ]:
x = "GATACCA"
y = "NNNNNNNNNNGATTACANNNNNNNNGATNNNNNNNNN"

# LOCAL mode finds the best matching region without aligning both sequences end-to-end.
align = aln.make_align(mode="local")
x_aln, y_aln, warp_path_x, warp_path_y, scoring_mat = align(seq_a=x, seq_b=y)

# Plot the score matrix and overlay the chosen traceback path.
plt.figure(figsize=(15, 8))
plt.imshow(X=scoring_mat)
plt.xticks(ticks=range(len(y)), labels=y)
plt.yticks(ticks=range(len(x)), labels=x)
plt.plot(warp_path_y, warp_path_x, "w")
plt.show()

# Read Alignment

We can also simulate the alignment of many reads originating from e.g., a bulk DNA sequencing experiment.

### From alignment algorithms to a small sequencing experiment

Imagine we know a DNA **reference sequence** but a sequencing instrument gives us only short fragments, called **reads**. We will simulate those fragments, introduce substitution errors, find where each fragment came from, and then combine evidence across reads to call candidate SNPs.

```text
Reference → sample reads → add noise → local read alignment → pileup → candidate SNPs
```

Each of the next exercises adds one piece of this pipeline.

### ❓ Sampling reads from a reference sequence

**Task:** Randomly sample DNA reads from the provided `reference_sequence`. Each read must contain between `min_read_len` and `max_read_len` bases, and all reads must remain within the reference.

**Expected outcome:** A `reads` list of DNA strings. The original instruction says **5 reads**, while the original code sets `n_reads = 20`. This guided version **preserves the original code setting (20)**; change it to 5 if you want to match the sentence.

**Hints:**
- `random.randint(a, b)` chooses a random integer **including both endpoints**.
- `reference_sequence[start:end]` slices from `start` up to (but not including) `end`.
- Require `end <= len(reference_sequence)` before appending a read.
- `reads.append(...)` stores a read, and `range_acquired = True` stops the inner retry loop.

**What this teaches:** Random sampling, `while` loops, indexing/slicing, and why read boundaries must stay inside a reference.

**Check your understanding:** Why can an otherwise valid random starting position produce a read that is too short?

**Original exercise instruction (retained):**

> Simulate the generation of a number of reads, by sampling 5 reads from the reference sequence `GATTTGGGGTTCAAAGCAGTATCGATCAAATAGTAAATCCATTTGTTCAACTCACAGTTTA`.


In [ ]:
import random

reference_sequence = "GATTTGGGGTTCAAAGCAGTATCGATCAAATAGTAAATCCATTTGTTCAACTCACAGTTTA"

n_reads = 20  # Number of reads to generate (original code value).
min_read_len = 15  # Minimum allowed read length.
max_read_len = 25  # Maximum allowed read length.

reads = []  # Collect all sampled reads here.

while len(reads) < n_reads:
    range_acquired = False  # Has this read been sampled within reference bounds?
    while not range_acquired:
        # TODO 1: Randomly choose a read length and a starting position.
        # TODO 2: Calculate end = start + read_len.
        # TODO 3: Only append reference_sequence[start:end] if end is valid.
        # TODO 4: Set range_acquired = True after a valid read.
        # --- Solution ---
        # Replace `pass` with the following code:
        # read_len = random.randint(min_read_len, max_read_len)
        # start = random.randint(0, len(reference_sequence) - 1)
        # end = start + read_len
        # if end <= len(reference_sequence):
        #     reads.append(reference_sequence[start:end])
        #     range_acquired = True
        pass  # Placeholder: remove after implementing the TODO above.

print(reads)

### ❓ Adding noise to the sampled reads

**Task:** Implement `add_noise()` so each base may be randomly changed to a *different* DNA base; return both the new read and the number of changed bases.

**Expected outcome:** `add_noise(read, noiselessness)` returns `(noisy_read, n_noisy_bases)`. The read length stays unchanged. The provided loop prints how many bases changed in each read.

**Hints:**
- `for base in read:` visits each DNA character in turn.
- For this teaching version, choose a substitution when `random.randrange(noiselessness) == 0` (probability `1 / noiselessness`).
- `random.choice(alternatives)` picks a replacement; exclude the original base from `alternatives`.
- `noisy_read += ...` extends the new string; `n_noisy_bases += 1` counts a real substitution.
- This deliberately models **substitution errors only** (not insertions or deletions).

**What this teaches:** Writing a function with parameters and a return value, iterating over strings, conditional logic, and simulating sequencing errors.

**Check your understanding:** Why must we exclude the original base when counting substitutions?

**Original exercise instruction (retained):**

> Add some noise to the sampled reads.


In [ ]:
def add_noise(read: str, noiselessness: int = 4) -> tuple[str, int]:
    """Add noise to a read."""
    # Possible DNA bases for a replacement.
    choices = ["A", "C", "G", "T"]
    n_noisy_bases = 0
    noisy_read = ""
    for base in read:
        # Each iteration processes ONE base of the input read.
        # TODO: Decide randomly whether this base changes.
        # If it changes, append a DIFFERENT base and increment the counter.
        # Otherwise, append the original base.
        # --- Solution ---
        # Replace `pass` with:
        # if random.randrange(noiselessness) == 0:
        #     alternatives = [candidate for candidate in choices if candidate != base]
        #     noisy_read += random.choice(alternatives)
        #     n_noisy_bases += 1
        # else:
        #     noisy_read += base
        pass  # Placeholder: remove after implementing the TODO.
    return noisy_read, n_noisy_bases



# Apply the function to every sampled read.
noisy_reads = []
for read in reads:
    noisy_read, n_noisy_bases = add_noise(read=read, noiselessness=3)
    print(f"Added {n_noisy_bases:2} noisy base(s): {read:10s} -> {noisy_read:10s}")
    noisy_reads.append(noisy_read)

### ❓ Aligning reads

**Task:** Align each noisy read to the **full** reference using a **local** aligner. Save its mapping position and aligned sequence.

**Expected outcome:** `mapping_positions` and `aligned_reads` each contain one entry per read. The cell prints the mapping position, aligned read, clipped bases, reference subsequence, and alignment visualization.

**Hints:**
- `aln.make_align(mode="local")` creates the Smith–Waterman-style alignment function.
- `align(seq_a=read, seq_b=reference_sequence)` returns five items: `read_aln`, `ref_aln`, `warp_path_read`, `warp_path_ref`, and `scoring_mat`.
- `"".join(read_aln)` turns a character sequence into one string.
- **Clipped bases** are bases of the original read outside the selected local alignment.
- The original code uses `warp_path_ref[0]` as the start coordinate; this relies on the course module's path conventions. Inspect `alignment.py` if coordinates look unexpected.

**What this teaches:** Applying **local** alignment for read mapping, unpacking function outputs, distinguishing aligned characters from path coordinates, and interpreting clipping.

**Check your understanding:** Why is local alignment more appropriate here than aligning the entire reference end-to-end with each short read?

**Original exercise instruction (retained):**

> Use the `make_align()` function from the `alignment` module to generate an `align()` function to align the noisy reads locally to the reference sequence.


In [ ]:
mapping_positions = []  # Start index of each mapped read.
aligned_reads = []  # Alignment string for each mapped read.

for read in noisy_reads:
    # TODO: Create/use a local aligner and align this read to reference_sequence.
    # Unpack the FIVE returned values using the names given in Hints.
    # --- Solution ---
    # Uncomment these lines before running:
    # align = aln.make_align(mode="local")
    # read_aln, ref_aln, warp_path_read, warp_path_ref, scoring_mat = align(
    #     seq_a=read, seq_b=reference_sequence
    # )
    # NOTE: uncomment the solution before running this cell.

    # The course code takes the first reference-path coordinate as mapping start.
    mapping_position = warp_path_ref[0]
    # Join the aligned bases (including any internal gaps) into a string.
    aligned_read = "".join(read_aln)
    mapping_positions.append(mapping_position)
    aligned_reads.append(aligned_read)

    print(f"Read                      : {read}")
    print(f"Mapping position (0-based): {mapping_position}")
    print(f"Aligned read              : {aligned_read}")
    aligned_read_without_gaps = list(filter(lambda elem: elem != "-", read_aln))
    print(f"Aligned read w/o gaps     : {''.join(aligned_read_without_gaps)}")
    print(f"Number of clipped bases   : {len(read) - len(aligned_read_without_gaps)}")
    print(
        f"Reference subsequence     : "
        f"{reference_sequence[warp_path_ref[0] : (warp_path_ref[-1] + 1)]}"
    )

    plt.figure(figsize=(15, 8))
    plt.imshow(X=scoring_mat)
    plt.plot(warp_path_ref, warp_path_read, "w")
    plt.xticks(ticks=range(len(reference_sequence)), labels=reference_sequence)
    plt.yticks(ticks=range(len(read)), labels=read)
    plt.show()

We now have the following set of aligned reads:

In [ ]:
print(f"Reference: {reference_sequence}")
for i, (pos, seq) in enumerate(zip(mapping_positions, aligned_reads)):
    print(f"Read {i:4}: {' ' * pos}{seq}")

# Variant Calling

Variant calling is a crucial step in analyzing DNA sequence data to identify differences or variations in the genetic code of individuals.
This process involves comparing an individual's DNA sequence (in the form of aligned reads) to a reference sequence to pinpoint variations such as single nucleotide polymorphisms (SNPs), insertions, deletions, and structural alterations.

While there exist many sophisticated variant calling algorithms such as [GATK](https://gatk.broadinstitute.org)'s [HaplotypeCaller](https://gatk.broadinstitute.org/hc/en-us/articles/360037225632-HaplotypeCaller), we will here focus on SNPs only and perform a simple majority voting per locus.

### Pileup and majority vote: a tangible example

```text
Reference position:       0 1 2 3 4 5
Reference:                A T G C A T
Read 1:                   A T G T
Read 2:                       G T A T
Read 3:                       G C A T
                                 ^
At position 3, observed bases might be T, T, C.
Majority = T, reference = C → candidate C→T SNP.
```

**A candidate is not proof of a biological variant.** In our toy simulation, substitutions were introduced as sequencing noise, so a majority vote can produce false positives. Real variant callers additionally consider read quality, depth, mapping uncertainty, and genotype models.

### ❓ SNP calling via majority vote

**Task:** Place aligned read bases into **pileups** indexed by reference position. For each nonempty pileup, use the most frequent observed base as the majority; report a **candidate SNP** when it differs from the reference base.

**Expected outcome:** A table with `POS`, `REF`, `MAJ`, `SNP`, and `PILEUP`, followed by a list of candidate SNP positions. Random noise means exact results can differ between runs.

**Mini example:** Reference base `C`, pileup `TTCT` → majority `T` → candidate `C → T` SNP.

**Hints:**
- `enumerate(seq)` produces `(offset, base)` pairs.
- `pileups[pos + offset].append(base)` stores one base at one reference position.
- `Counter(pileup).most_common(1)[0][0]` returns the most frequent base.
- Skip empty pileups: **no coverage does not mean a SNP**.
- The supplied `pos + offset` shortcut is valid only when the aligned string can be treated as consecutive reference positions. An alignment with indels needs explicit reference-coordinate tracking.

**What this teaches:** Building a simple read pileup, counting with `Counter`, majority voting, and why low coverage / sequencing errors can produce false candidate variants.

**Check your understanding:** If just one read covers a position, how confident should you be about a mismatch?

**Original exercise instruction (retained):**

> Complete the following code to call SNPs via majority voting.


In [ ]:
from collections import Counter

pileups = [[] for _ in range(len(reference_sequence))]  # One list per reference base.
snps = {}  # Store candidate variants as {position: alternative_base}.

# Accumulate the pileups.
for pos, seq in zip(mapping_positions, aligned_reads):
    # TODO: Iterate over aligned bases with enumerate(seq).
    # Add each non-gap base to the matching reference-position pileup.
    # --- Solution ---
    # Replace `pass` with:
    # for offset, base in enumerate(seq):
    #     if base != "-" and pos + offset < len(reference_sequence):
    #         pileups[pos + offset].append(base)
    pass  # Placeholder: remove after implementing the TODO.

# SNP calling via majority voting.
print("POS   REF   MAJ   SNP   PILEUP")
for i, pileup in enumerate(pileups):
    majority_base = ""
    snp = "-"

    # Make a pileup string.
    pileup = "".join(pileup)

    # We can only call variants for non-empty pileups.
    if pileup:
        # TODO: Find the most frequent base in this nonempty pileup.
        # Store the character (not the count) in majority_base.
        # --- Solution ---
    # Replace `pass` with:
        # majority_base = Counter(pileup).most_common(1)[0][0]
        pass  # Placeholder: remove after implementing the TODO.

    # Call it if it's a SNP.
    if pileup and majority_base not in ("", "-") and majority_base != reference_sequence[i]:
        snp = majority_base
        snps[i] = snp

    print(f"{i:3}   {reference_sequence[i]:3}   {majority_base:3}   {snp:3}   {pileup}")

In [ ]:
# List all SNPs.
print("POS   SNP")
for pos, snp in snps.items():
    print(f"{pos:3}   {snp:3}")

### End-to-end teaching checkpoint

**Follow one read through the workflow:** sample a short sequence → change a few bases → align it to the reference → use the mapped start to build a pileup → compare the majority base with the reference.

**Important distinction:** A simulated sequencing substitution is an *error*, while a biological SNP is a *real sample–reference difference*. This toy workflow cannot reliably tell them apart without more evidence.


## Recap — What should students be able to explain?

| Concept | One-sentence explanation |
|---|---|
| Needleman–Wunsch | Find the best **end-to-end** alignment. |
| `F(i,j)` | Best score for aligning the first `i` and `j` sequence characters. |
| Three recurrence terms | Choose match/mismatch, gap in Y, or gap in X. |
| Traceback | Follow stored best-choice arrows backward to recover aligned letters. |
| Smith–Waterman | Find the best **local** matching region; scores can restart at 0. |
| `x_aln` vs path | Aligned letters versus alignment/grid coordinates. |
| Read mapping | Identify where a short read fits within a reference. |
| Pileup | Collect bases from overlapping reads at each reference position. |
| Majority vote | Report a **candidate** substitution when the most frequent observed base differs from the reference. |

**Instructor discussion prompts:** What happens if two traceback choices tie? What if most reads are wrong at the same site? What if a read maps equally well to two reference locations? What changes if reads contain insertions or deletions?

